# Numerical error in solving linear equations

Examine the residual after solving $Ax = b$

Residual: $r = Ax-b$. This is supposed to be small.

However, when a matrix $A$ is **ill-conditioned**, the solution $x=A^{-1}b$ can be numerically unstable. We can observe the condition of a matrix by computing the condition number of $A$, $\kappa(A)$. If $\kappa(A)$ is low (lower than 1,000), the matrix is well-conditioned, and if it's too high, the matrix is ill-conditioned.

Keyword for further reading: **condition number of a matrix**

In [ ]:
import numpy as np

n = 500

# case 1: draw from Normal(0,1)
# A = np.random.randn(n,n)

# case 2: A is lower triangular by stripping Normal(0,1) in lower part
# Intentionally, add the diagonal elements as Normal(10,1)
# A = np.tril( np.random.randn(n,n), k=-1 ) # off-diagonal element
# np.fill_diagonal(A, 10 + np.random.randn(n)) # add diagonal element as 10 + U[0,1]

# case 3: A is lower triangular by stripping Normal(0,1) in lower/upper part
# This case can lead to ill-conditioned because the diagonal of A are N(0,1)
A = np.triu( np.random.randn(n,n))
# A = np.tril( np.random.randn(n,n))

b = np.random.randn(n)

x = np.linalg.solve(A,b)
r = np.linalg.norm( A @ x - b )

condnum = np.linalg.cond(A)
print(f"Condition number of A is {condnum:.2f}")
print(f"Residual norm is {r:.2e}")


Condition number of A is 11682281470536159232.00
Residual norm is 4.92e+139


# Solve $Ax=b$ when $A$ has special structure.

## Diagonal system.

Solve $Dx = b$. Typically, the information in $D$ is only stored as the diagonal element $d=(d_1,d_2,\ldots,d_n)$.

The solution is $x=D^{-1}b $ where $x_k = b_k/d_k$ for $k=1,2,\ldots,n$.

In [ ]:
import numpy as np
import time

n = 1000
d = np.random.randn(n)
b = np.random.randn(n)

# A: recommended
t1 = time.perf_counter()
x1 = b/d
t2 = time.perf_counter()
dt1 = (t2-t1)*1e3 # msec

# B: slower method
t1 = time.perf_counter()
D = np.diag(d)
x2 = np.linalg.solve(D,b)
t2 = time.perf_counter()
dt2 = (t2-t1)*1e3 # msec

print(f"Are the two solutions close? {np.allclose(x1,x2)}")
print(f"Computation time method A: {dt1:.4f} msec and method B: {dt2:.4f} msec")

Are the two solutions close? True
Computation time method A: 1.3043 msec and method B: 96.0919 msec


## Orthogonal system

Solve $Ax =b $ when $A$ is orthogonal.

$A$ is orthogonal iff $A^TA = AA^T = I$. Hence, $A^{-1} = A^T$. The solution $x = A^{-1}b = A^T b$.

In [ ]:
import numpy as np
import time

# Generate data
def gen_orthogonal(n):
  H = np.random.randn(n, n)
  Q, _ = np.linalg.qr(H)  # return QR factorization where Q is orthogonal
  return Q

n = 1000
A = gen_orthogonal(n)
b = np.random.randn(n)

# A: recommended
t1 = time.perf_counter()
x1 = A.T @ b
t2 = time.perf_counter()
dt1 = (t2-t1)*1e3 # msec

# B: slower method (just use general solver, did not take A's structure)
t1 = time.perf_counter()
x2 = np.linalg.solve(A,b)
t2 = time.perf_counter()
dt2 = (t2-t1)*1e3 # msec

print(f"Are the two solutions close? {np.allclose(x1,x2)}")
print(f"Computation time method A: {dt1:.4f} msec and method B: {dt2:.4f} msec")

Are the two solutions close? True
Computation time method A: 18.8002 msec and method B: 4403.4746 msec


## Permutation system

Solve $Px = b$ when $P$ is permutation.

Typically, the information of $P$ is given as an array of indices corresponding to the permutation order.

For example, when $n=3$ and the index is $(2,3,1)$, the matrix $P$ is
$$
P = \begin{bmatrix}
0 & 1 & 0 \\
0 & 0 & 1 \\
1 & 0 & 0
\end{bmatrix}
$$
We don't have to store $P$, just store the index vector. Therefore, the task is that given the permutation index and vector $b$, solve the linear system.

**Solution:** As permutation is also orthogonal, the solution is $x=P^Tb$. Moreover, $P^T$ can be read easily from the permutation index. From the example,
$$
P^T =
\begin{bmatrix}
0 & 0 & 1 \\
1 & 0 & 0 \\
0 & 1 & 0
\end{bmatrix}
$$
and the permutation index of $P^T$ is $(3,1,2)$ which is obtained by find the position of $(1,2,3)$ in the permutation index of $P$. This can be done by sorting the permutation index of $P$ and then return the sorting indices.

In [ ]:
import numpy as np

n = 1000
idx = np.random.permutation(n)
b = np.random.randn(n)

# A: recommended
t1 = time.perf_counter()
idx_Pt = np.argsort(idx)
x1 = b[idx_Pt]
t2 = time.perf_counter()
dt1 = (t2-t1)*1e3 # msec

# B: slower method
t1 = time.perf_counter()
I = np.eye(n)
P = I[idx,:]
# x2 = np.linalg.solve(P,b)
x2 = P.T @ b
t2 = time.perf_counter()
dt2 = (t2-t1)*1e3 # msec

res1 = np.linalg.norm( P@x1 - b)
res2 = np.linalg.norm( P@x2 - b)

print(f"Are the two solutions close? {np.allclose(x1,x2)}")
print(f"Residual error of the solution A: {res1:.4f} and B: {res2:.4f}")
print(f"Computation time method A: {dt1:.4f} msec and method B: {dt2:.4f} msec")


Are the two solutions close? True
Residual error of the solution A: 0.0000 and B: 0.0000
Computation time method A: 4.4036 msec and method B: 152.7897 msec


## Triangular system

Solve $Lx = b$, where $L$ is lower triangular.

Fact: Solving a triangular system uses only $O(n^2)$ flops

How to implement it?

1. `np.linalg.solve` (generate solver, performs LU with pivoting and then solve it, require $O(n^3) $.
2. `scipy.linalg.solve_traingular` (use when A is already known to be triangular, skips the factorization, and goes straight to substitution step, involve $O(n^2)$.

In [ ]:
import numpy as np
from scipy.linalg import solve_triangular
import time

n = 5000

# Generate matrix data
L = np.tril( np.random.randn(n,n), k=-1 ) # off-diagonal element
np.fill_diagonal(L, 10+ np.random.randn(n)) # add diagonal element Normal(10,1)

U = np.triu( np.random.randn(n,n), k=1)
np.fill_diagonal(U, 10 + np.random.randn(n)) # add diagonal element Normal(10,1)

b = np.random.randn(n)

# ---- Solving Lower system
t1 = time.perf_counter()
x1 = np.linalg.solve(L,b)
t2 = time.perf_counter()
dtlower = [(t2-t1)*1e3]  # msec

t1 = time.perf_counter()
x2 = solve_triangular(L,b, lower=True)
t2 = time.perf_counter()
dtlower.append((t2-t1)*1e3)
reslower = np.array([ np.linalg.norm(L @ x1 - b), np.linalg.norm(L @ x2 - b)])

print("\nSolving lower triangular system")
if np.allclose(x1,x2):
  print("Two solutions are close")
else:
  print(f"Two solutions are different: {np.linalg.norm(x1-x2)}")

print(f"Computation time: np.linalg.solve vs solve_triangular {np.round(dtlower,4)} msec")
print(f"Residual for np.linalg.solve vs solve_triangular:  {reslower}")


# ----- Solving Upper system
t1 = time.perf_counter()
z1 = np.linalg.solve(U,b)
t2 = time.perf_counter()
dtupper = [(t2-t1)*1e3]  # msec

t1 = time.perf_counter()
z2 = solve_triangular(U,b)
t2 = time.perf_counter()
dtupper.append((t2-t1)*1e3)
resupper = np.array([ np.linalg.norm(U @ z1 - b), np.linalg.norm(U @ z2 - b)])

print("\nSolving upper triangular system")
print(f"Check if two solutions are close: {np.allclose(z1,z2)}")
print(f"Computation time: np.linalg.solve vs solve_triangular {np.round(dtupper,4)} msec")
print(f"Residual for np.linalg.solve vs solve_triangular:  {resupper}")



Solving lower triangular system
Two solutions are close
Computation time: np.linalg.solve vs solve_triangular [2177.1567   20.6546] msec
Residual for np.linalg.solve vs solve_triangular:  [0.00130217 0.00062578]

Solving upper triangular system
Check if two solutions are close: True
Computation time: np.linalg.solve vs solve_triangular [2133.979   20.048] msec
Residual for np.linalg.solve vs solve_triangular:  [0.00155403 0.00174634]


# LU factorization
for a square $A$, LU factorization is
$$
A = PLU
$$
where
1. $P$ is permutation (for partial pivoting)
2. $L$ is lower triangular with unit diagonal elements
3. $U$ is upper triangular

To understand the concept, we use two methods based on Scipy (float computation) and SymPy (for study only).

1. `scipy.linalg.lu`: partial pivoting relies on the rule that swapping rows to find the largest absolute value
2. SymPY: `LUdecomposition`: only swaps rows if the current pivot is exactly zero.

Both approaches shown here as to illustrate the decomposition for our study. For efficient implementation in practice, see the next topic.

Both methods rely on different pivoting strategies, so their results can be different.

In [ ]:
import numpy as np
import sympy as sp
from scipy.linalg import lu, lu_factor, lu_solve, solve_triangular
import time

n = 5

# A = np.random.randint(-10,10,(n,n))
A = np.array([[2,4,-2,2],[4,9,-3,8],[-2,-1,1,10],[2,2,3,7]])

# Use Scipy (for numerical practice)
P,L,U = lu(A)
A1 = P @ L @ U

# Use SymPY (to see symbolic result)
Asym = sp.Matrix(A)
Ls, Us, perm = Asym.LUdecomposition()

print(f"Results of Scipy: \n")
print(f"A is \n {A}")
print(f"P is \n {P}")
print(f"L is \n {L}")
print(f"U is \n {U}")
print(f"Check if reconstructed A is close to A: {np.allclose(A1,A)} \n")

print(f"Results of SymPy: \n")
print("Lower Triangular L:")
sp.pprint(Ls)

print("\nUpper Triangular U:")
sp.pprint(Us)

sp.pprint(perm)


Results of Scipy: 

A is 
 [[ 2  4 -2  2]
 [ 4  9 -3  8]
 [-2 -1  1 10]
 [ 2  2  3  7]]
P is 
 [[0. 0. 0. 1.]
 [1. 0. 0. 0.]
 [0. 1. 0. 0.]
 [0. 0. 1. 0.]]
L is 
 [[ 1.          0.          0.          0.        ]
 [-0.5         1.          0.          0.        ]
 [ 0.5        -0.71428571  1.          0.        ]
 [ 0.5        -0.14285714 -0.13793103  1.        ]]
U is 
 [[ 4.          9.         -3.          8.        ]
 [ 0.          3.5        -0.5        14.        ]
 [ 0.          0.          4.14285714 13.        ]
 [ 0.          0.          0.          1.79310345]]
Check if reconstructed A is close to A: True 

Results of SymPy: 

Lower Triangular L:
⎡1   0    0    0⎤
⎢               ⎥
⎢2   1    0    0⎥
⎢               ⎥
⎢-1  3    1    0⎥
⎢               ⎥
⎣1   -2  -7/4  1⎦

Upper Triangular U:
⎡2  4  -2  2 ⎤
⎢            ⎥
⎢0  1  1   4 ⎥
⎢            ⎥
⎢0  0  -4  0 ⎥
⎢            ⎥
⎣0  0  0   13⎦
[]


In [ ]:
P


array([[0., 0., 0., 1.],
       [1., 0., 0., 0.],
       [0., 1., 0., 0.],
       [0., 0., 1., 0.]])

# CONFUSION with LU factorization format

1. `scipy.linalg.lu`: returns $P,L,U$ for $A = PLU$
2. `scipy.linalg.lu_factor`: returns a list of $L,U$ as one matrix and the pivoting array (piv)that keeps the history of swapping (not the permutation index associated with $P$ in `scipy.linalg.lu).

For example, if piv = (2,2,3), it means

1. at step 1: swap row 1 with row 2.
2. at step 2: swap row 2 with row 2 (no change).
3. at step 3: swap row 3 with row 3 (no change).


In [ ]:
import numpy as np
from scipy.linalg import lu_factor

A = np.random.randint(-10,10,(n,n))

# .lu
P,L,U = lu(A)

# .lu_factor
lu_and_piv = lu_factor(A)
print(lu_and_piv[:])

# This function converts pivoting array from lu_factor to match P returned by .lu
def piv_to_P(piv):
    n = len(piv)
    P = np.eye(n)
    for i, p in enumerate(piv):
        # Swap rows i and p sequentially
        P[[i, p], :] = P[[p, i], :]
    return P

piv = lu_and_piv[-1]
P_from_piv = piv_to_P(piv)
print(np.allclose(P_from_piv, P.T))

(array([[-10.        ,  -3.        ,   4.        ,  -4.        ,
          0.        ],
       [  0.9       ,   7.7       ,  -8.6       ,  10.6       ,
          6.        ],
       [  0.9       ,   0.48051948,   3.53246753,   5.50649351,
          5.11688312],
       [  1.        ,   0.25974026,  -0.5       ,  12.        ,
         -7.        ],
       [  0.3       ,   0.11688312,   0.22794118,  -0.19117647,
         -3.20588235]]), array([2, 1, 2, 3, 4], dtype=int32))
True


# How to use LU factorization?

$ PLUx = b$ splits to
1. $LUx = P^Tb$
2. $Lz = P^Tb$ ( by letting $z = Ux$)
3. $Ux = z$

**Implementation 1:**
1. `scipy.linalg.lu`: Return P,L,U in LU decomposition
2. `scipy.linalg.solve_triangular`: Solve a triangular system

**Implementation 2:**
1. `scipy.linalg.lu_factor`: Return LU factors and pivoting indices
2. `scipy.linalg.lu_solve`: Solve the linear system using the given LU factorization

In [ ]:
import numpy as np
from scipy.linalg import lu, lu_factor, lu_solve, solve_triangular
import time

n = 5000

A = np.random.randn(n,n)
b = np.random.randn(n)

dt = []
res = []
# Implementation 1:
t1 = time.perf_counter()
P,L,U = lu(A)
z = solve_triangular(L,P.T @ b, lower = True)
x1 = solve_triangular(U,z)
t2 = time.perf_counter()
dt.append( (t2-t1)*1e3 )  # msec
res.append( np.linalg.norm( A @ x1 - b))

# Implementation 2:
t1 = time.perf_counter()
lu_and_piv = lu_factor(A)
x2 = lu_solve(lu_and_piv, b)
t2 = time.perf_counter()
dt.append( (t2-t1)*1e3 )  # msec
res.append( np.linalg.norm( A @ x2 - b))

print(f"Computation time: 1 vs 2 : {np.round(dt,3)} msec")
print(f"Residual of solution: 1 vs 2 : {np.round(res,3)}")


Computation time: 1 vs 2 : [5064.208 2968.599] msec
Residual of solution: 1 vs 2 : [0. 0.]


# Solving a square system $Ax = b$ when $A$ is fixed but $b$ changes.

Solve $Ax_k = b_k$ for $k=1,2,\ldots,M$ where $M$ is the number of vector $b_k$'s. Here, $b_k \in \mathbf{R}^n$ (is a vector, not scalar)

Math: We can stack $b_k$'s as columns of $B$.

$$
B = \begin{bmatrix} b_1 & b_2 & \cdots & b_M \end{bmatrix}
$$
and the solution would be $x_k = A^{-1} b_k$, so we can stack $x_k$ in the $k$th column of $X$

$$
\begin{bmatrix} x_1 & x_2 & \cdots & x_M \end{bmatrix} = \begin{bmatrix} A^{-1} b_1 & A^{-1} b_2 & \cdots & A^{-1} b_N \end{bmatrix} \quad \Leftrightarrow \quad X = A^{-1} B
$$

**Implementations:**
1. `np.linalg.solve` (Perform LU and solve the system by calling Fortran (low-level) library)
2. `scipy.linalg.lu` and `scipy.linalg.solve_triangular` (Use LU factors and then exploits the LU factors)
Then, it may creates many objects: $P,L,U$ and calling high-level python library when solving triangular system.
3. `scipy.linalg.lu_factor` and `scipy.linalg.lu_solve` (Use LU factors and solve the triangular system in one go using low-level library )



## When $b_k$'s are **known at the same time**.

This means the whole block $B = \begin{bmatrix} b_1 & b_2 & \cdots & b_M \end{bmatrix}$ is given at once.

**Recommendation**
Please observe the coding results below.

`np.linalg.solve` is typically faster as it has the loweset overhead, and manages memory and CPU cache in a single pass. In this case, we assume all of $b_k$'s are **known at once**

In [ ]:

import numpy as np
from scipy.linalg import lu, lu_factor, lu_solve, solve_triangular
import time

n,M = 1000, 1000

A = np.random.randn(n,n)
B = np.random.randn(n,M) # stack each b in the column of B

# Implementation 1: using np.linalg.solve
t1 = time.perf_counter()
X = np.linalg.solve(A,B)
t2 = time.perf_counter()
dt = (t2-t1)*1e3  # msec
print(f"Computation time: using np.linalg.solve  is {dt:.4f} msec")

# Implementation 2: Using LU  decomposition #
# PLUx = B,  LUx = P^TB , Lz = P^TB,  Ux = z
# Let the system know that L,U are triangular
t1 = time.perf_counter()
P,L,U = lu(A)
Z = solve_triangular(L,P.T@B, lower = True)
X2 = solve_triangular(U,Z)
t2 = time.perf_counter()
dtlu = (t2-t1)*1e3  # msec
print(f"Computation time: using LU is {dtlu:.4f} msec")

# Implementation 3: Using LU factors #
# PLUx = B,  LUx = P^TB , Lz = P^TB,  Ux = z
# LU_SOLVE() handles solving triangular system (the steps above) for us
t1 = time.perf_counter()
lu_and_piv = lu_factor(A)
X3 = lu_solve(lu_and_piv,B)
t2 = time.perf_counter()
dtlufac = (t2-t1)*1e3  # msec
print(f"Computation time: using LU factor and LU_solve  is {dtlufac:.4f} msec")

Computation time: using np.linalg.solve  is 145.5771 msec
Computation time: using LU is 460.6351 msec
Computation time: using LU factor and LU_solve  is 185.3315 msec


## When $b_k$'s **arrive sequentially**.

Suppose $B = \begin{bmatrix} b_1 & b_2 & \cdots & b_M \end{bmatrix}$, but you can only read $b_k$ one at a time, and you aim to solve $Ax_k = b_k$ for $k=1,2,\ldots,M$.

How to implement it ?

1. Strategy A: `np.linalg.solve` $O(n^3)\times M$
2. Stratgey B: `'np.linalg.lu_factor` and `np.linalg.lu_solve` ($\frac{2}{3}n^3) + M\cdot 2n^2$)

**Recommendation**
Please observe the coding results below.

`scipy.linalg.lu_factor` + `scipy.linalg.lu_solve` is fastest as it highy optimized

In [ ]:
import numpy as np
from scipy.linalg import lu_factor, lu_solve
import time

# Problem Setup
n = 500
num_steps = 2000
A = np.random.randn(n, n)
# Generate num_steps sequential 'b' vectors
B_sequence = [np.random.randn(n) for _ in range(num_steps)]

# --- Strategy A: Naive Sequential Solve ---
t_start = time.perf_counter()
for b_t in B_sequence:
    # TODO: Solve Ax = b_t using the standard NumPy solver
    x_t = np.linalg.solve(A, b_t)
t_end = time.perf_counter()
print(f"Strategy A Total Time: {t_end - t_start:.4f} seconds")

# --- Strategy B: Efficient LU Solve ---
t_start = time.perf_counter()
# TODO: Step 1 - Pre-compute LU factors outside the loop
lu_piv = lu_factor(A)

for b_t in B_sequence:
    # TODO: Step 2 - Solve using the pre-computed factors
    x_t = lu_solve(lu_piv, b_t)
t_end = time.perf_counter()
print(f"Strategy B Total Time: {t_end - t_start:.4f} seconds")

Strategy A Total Time: 12.0640 seconds
Strategy B Total Time: 0.2911 seconds


Results of SymPy: 

Lower Triangular L:
⎡ 1   0   0⎤
⎢          ⎥
⎢1/2  1   0⎥
⎢          ⎥
⎣ 1   -3  1⎦

Upper Triangular U:
⎡2  4  6 ⎤
⎢        ⎥
⎢0  1  2 ⎥
⎢        ⎥
⎣0  0  -1⎦
[]


# Cholesky factorization
for $A \succ 0$, the Cholesky factorization is
$$
A = LL^T
$$
where $L$ is lower triangular matrix with nonzero diagonals.

To understand the concept, we use two methods based on Scipy (float computation) and SymPy (for study only).

1. `numpy.linalg.cholesky`
2. `scipy.linalg.cholesky`
3. SymPY: `Cholesky`

In [24]:
import numpy as np
import scipy.linalg
import sympy
import time

# 1. Generate a random nxn SPD matrix A
n = 5

X = np.random.randint(-3, 3, size=(n, n))
A = X @ X.T + np.eye(n) # Ensuring it's SPD

print(f"Original Matrix A:\n{A}\n")

# --- 1) NumPy Implementation ---
start = time.perf_counter()
L_np = np.linalg.cholesky(A)
end = time.perf_counter()
print("1) NumPy Implementation")
print(f"L:\n{L_np.round(3)}")
print(f"Time: {(end - start)*1e3:.8f} m-seconds\n")
print(f"Decomposition is correct: {np.allclose(A, L_np @ L_np.T)}")

# --- 2) SciPy Implementation ---
# By default scipy returns Upper (U), set lower=True for L
start = time.perf_counter()
L_scipy = scipy.linalg.cholesky(A, lower=True)
end = time.perf_counter()
print("2) SciPy Implementation")
print(f"L:\n{L_scipy.round(3)}")
print(f"Time: {(end - start)*1e3:.8f} m-seconds\n")

# --- 3) SymPy Implementation ---
A_sympy = sympy.Matrix(A)
start = time.perf_counter()
L_sympy = A_sympy.cholesky()
end = time.perf_counter()
print("3) SymPy Implementation")
sympy.pprint(L_sympy)
print(f"Time: {(end - start)*1e3:.8f} m-seconds\n")

# --- 4) Scipy.linalg.cho_factor ---
c, lower = scipy.linalg.cho_factor(A)


Original Matrix A:
[[ 14. -10.   4.  17.   6.]
 [-10.  13.   4. -14.  -4.]
 [  4.   4.  14.   3.   4.]
 [ 17. -14.   3.  30.   5.]
 [  6.  -4.   4.   5.  16.]]

1) NumPy Implementation
L:
[[ 3.742  0.     0.     0.     0.   ]
 [-2.673  2.42   0.     0.     0.   ]
 [ 1.069  2.833  2.198  0.     0.   ]
 [ 4.543 -0.767  0.144  2.958  0.   ]
 [ 1.604  0.118  0.888 -0.786  3.465]]
Time: 0.85208300 m-seconds

Decomposition is correct: True
2) SciPy Implementation
L:
[[ 3.742  0.     0.     0.     0.   ]
 [-2.673  2.42   0.     0.     0.   ]
 [ 1.069  2.833  2.198  0.     0.   ]
 [ 4.543 -0.767  0.144  2.958  0.   ]
 [ 1.604  0.118  0.888 -0.786  3.465]]
Time: 0.80251200 m-seconds

3) SymPy Implementation
⎡3.74165738677394           0                   0                  0           ↪
⎢                                                                              ↪
⎢-2.67261241912424   2.42015347801392           0                  0           ↪
⎢                                                

# Solve linear system using Cholesky

For $A=LL^T$, we can solve $Ax = b$ in two steps:

1. $Lz = b$
2. $L^Tx = z$

The cost is $\mathcal{O}(1/3n^3) + 2n^2$ which is the factorization and solve step.

Implementation:
1. `c, low = scipy.linalg.cho_factor` returns the Cholesky factor $c$ stored in upper or lower triangle part which can be checked from `low` output. If `low = True` it means the factor is in the lower part.
2. `scipy.linalg.cho_solve` Use the factor from `scipy.linalg.cho_factor` to solve the linear equation $Ax=b$.

In [25]:
import numpy as np
import scipy.linalg
import time

# 1. Setup parameters
n = 500
num_b = 1000

# 2. Generate Tri-band SPD Matrix A
# Main diagonal (d), upper diagonal (u), and lower diagonal (l)
d = np.ones(n) * 4  # Large diagonal ensures positive definiteness
u = np.ones(n-1) * 1
l = np.ones(n-1) * 1

A = np.diag(d) + np.diag(u, k=1) + np.diag(l, k=-1)

# Generate 1000 random b vectors
B = np.random.randn(n, num_b)

# --- Approach A: Cholesky Factorization (Pre-computing) ---
# We factorize A once, then solve for each b
start_cho = time.perf_counter()

# Factorize: returns (c, low) where c is the matrix and low is a boolean
c, low = scipy.linalg.cho_factor(A)

cho_solutions = np.zeros_like(B)
for i in range(num_b):
    cho_solutions[:, i] = scipy.linalg.cho_solve((c, low), B[:, i])

end_cho = time.perf_counter()
time_cho = end_cho - start_cho

# --- Approach B: General Solver (np.linalg.solve) ---
# General solvers usually perform LU decomposition or similar every single time
start_gen = time.perf_counter()

gen_solutions = np.zeros_like(B)
for i in range(num_b):
    gen_solutions[:, i] = np.linalg.solve(A, B[:, i])

end_gen = time.perf_counter()
time_gen = end_gen - start_gen

# 3. Output Results
print(f"Matrix Size: {n}x{n}")
print(f"Number of solves: {num_b}\n")
print(f"Cholesky (cho_factor + loop cho_solve): {time_cho:.4f} seconds")
print(f"General Solver (loop np.linalg.solve):  {time_gen:.4f} seconds")
print(f"Speedup Factor: {time_gen / time_cho:.2f}x")

# Verification: Ensure results are identical
max_diff = np.max(np.abs(cho_solutions - gen_solutions))
print(f"\nMax difference between solutions: {max_diff:.2e}")

Matrix Size: 500x500
Number of solves: 1000

Cholesky (cho_factor + loop cho_solve): 0.6958 seconds
General Solver (loop np.linalg.solve):  5.5233 seconds
Speedup Factor: 7.94x

Max difference between solutions: 6.66e-16
